Imports

In [1]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-16 15:32:30.670071: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-16 15:32:32.071621: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [33]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=4:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)
cluster.scale(jobs=1)

In [34]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [35]:
#model=scm.fit(client,dat,nb_formula="umis_mpra_bc ~ C(cre_id)-1",zi_formula="C(rep_id)",broken_on="unified",dry=False)
model=scm.fit(client,dat,nb_formula="umis_mpra_bc ~ C(cre_id)-1",zi_formula="C(rep_id)-1",broken_on="cell_type",dry=False)

In [36]:
unif=scm.fit(client,dat,nb_formula="umis_mpra_bc ~ C(cre_id)*C(cell_type)-1",zi_formula="C(rep_id)-1",broken_on="unified",dry=False)

In [37]:
unif.result()

In [38]:
model.result()

In [39]:
def model_to_parameters(model):
    """
    """
    #currently assuming a single theta per model (equal variances within a model).
    #also does not intelligently sum for interaction effects : so just use for non-interaction for now...
    #Need also to test with incomplete matricies (non-cartesian product) : applying labels will probably break

    #recall the order within each tuple: (X,y,Z), but here we skip y
    #so it is just X, Z
    
    zi={}
    nb={}
    theta={}

    for key in model.model:
        ## theta ##
        theta[key]=model.model[key]['weights']['theta'].squeeze()
        
        ## ZI ##
        #extract min design matrix & sort
        #We need to sort to match the order in the tensorzinb model...
        zi_df=model.uniq_predictor[key][1]
        zi_df=zi_df.sort_values(zi_df.columns.tolist(),ascending=False)

        #multiply out & undo link
        result=(zi_df.values @ model.model[key]["weights"]["x_pi"]).squeeze()
        result=1/(1+np.exp(-result))
        result=pd.Series(result,index=model.uniq_predictor[key][1].columns)
        zi[key]=result

        ## NB ##
        #extract min design matrix & sort
        nb_df=model.uniq_predictor[key][0]
        nb_df=nb_df.sort_values(nb_df.columns.tolist(),ascending=False)

        #multiply out & undo link
        result=(nb_df.values @ model.model[key]['weights']['x_mu']).squeeze()
        result=np.exp(result)
        result=pd.Series(result,index=model.uniq_predictor[key][0].columns)
        nb[key]=result

    #patch up the dataframes and return them.

    zi=pd.DataFrame(zi)
    zi.columns.name="zero inflation fraction"

    nb=pd.DataFrame(nb)
    nb.columns.name="mean parameter"

    theta=pd.Series(theta)
    #theta.columns.name="theta dispersion"
    
    return (nb,zi,theta)


#nb,zi,theta=model_to_parameters(model.result())

In [40]:
nb,zi,theta=model_to_parameters(unif.result())

In [51]:
working=unif.result().uniq_predictor['unified'][0]
working=working.sort_values(working.columns.to_list(),ascending=False)

In [54]:
working

,C(cre_id)[everybody],C(cre_id)[neurogene],C(cre_id)[nobody],C(cre_id)[redgene],C(cre_id)[somebody],C(cell_type)[T.brain],C(cre_id)[T.neurogene]:C(cell_type)[T.brain],C(cre_id)[T.nobody]:C(cell_type)[T.brain],C(cre_id)[T.redgene]:C(cell_type)[T.brain],C(cre_id)[T.somebody]:C(cell_type)[T.brain]
904,1,0,0,0,0,1,0,0,0,0
3266,1,0,0,0,0,0,0,0,0,0
1798,0,1,0,0,0,1,1,0,0,0
4262,0,1,0,0,0,0,0,0,0,0
0,0,0,1,0,0,1,0,1,0,0
2263,0,0,1,0,0,0,0,0,0,0
1355,0,0,0,1,0,1,0,0,1,0
3767,0,0,0,1,0,0,0,0,0,0
440,0,0,0,0,1,1,0,0,0,1
2762,0,0,0,0,1,0,0,0,0,0


In [ ]:
working.columns[]

In [31]:
cluster.close()